In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time

logging.basicConfig(
    filename="logs/ingestion_db.log",
    level = logging.DEBUG,
    format ="%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a",
    force = True
)

engine= create_engine('sqlite:///inventory.db')

def ingest_db(df,table_name, engine, mode='replace'):
    '''this fuction will ingest the dataframe into data tabel'''
    df.to_sql(table_name , con =engine , if_exists= mode, index = False)

def load_raw_data():
    '''this function will load csv as dataframe and ingest it into db'''
    start = time.time()
    
    # Files already successfully ingested based on log
    skip_files = ['begin_inventory.csv', 'end_inventory.csv', 'purchases.csv', 'purchase_prices.csv']
    
    for file in os.listdir('data'):
        if '.csv' in file:
            if file in skip_files:
                logging.info(f'Skipping already ingested file: {file}')
                continue
                
            logging.info(f'Ingesting {file} in db')
            
            chunk_iter = pd.read_csv('data/'+file, chunksize=100000)
            first_chunk = True
            
            for df in chunk_iter:
                mode = 'replace' if first_chunk else 'append'
                ingest_db(df, file[:-4],engine, mode)
                first_chunk = False
                
    end = time.time()
    total_time = (end - start)/60
    logging.info('----------Ingestion Complate----------')
    logging.info(f'\nTotal Time Taken : {total_time} Minutes')

if __name__ == '__main__':
    load_raw_data()

In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging
import time
logging.basicConfig(
    filename="logs/ingestion_db.log",
    level = logging.DEBUG,
    format ="%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a",
    force = True
)

engine= create_engine('sqlite:///inventory.db')

def ingest_db(df,table_name, engine):
    '''this fuction will ingest the dataframe into data tabel'''
    df.to_sql(table_name , con =engine ,  if_exists= 'replace', index = False)
    
def load_raw_data():
    '''this function will load csv as dataframe and ingest it into db'''
    start = time.time()
    for file in os.listdir('data'):
        if '.csv' in file:
               df= pd.read_csv('data/'+file)
               logging.info(f'Ingesting {file} in db')
               ingest_db(df, file[:-4],engine)
    end = time.time()
    total_time = (end - start)/60
    logging.info('----------Ingestion Complate----------')
    logging.info(f'\nTotal Time Taken : {total_time} Minutes')

if __name__ == '__main__':
    load_raw_data()

In [4]:
import pandas as pd
from sqlalchemy import create_engine

# Connect to your database
engine = create_engine('sqlite:///inventory.db')

# Ask the database to count the rows in the sales table
query = "SELECT COUNT(*) FROM begin_inventory;"

# Run the query and print the result
row_count = pd.read_sql(query, engine)
print(row_count)

   COUNT(*)
0    206529


(206529, 9)
(224489, 9)
(2372474, 16)
(12261, 9)
(12825363, 14)
(5543, 10)


In [3]:
# Grab just the first 5 rows to verify the columns and data
sample_data = pd.read_sql("SELECT * FROM sales LIMIT 5;", engine)
sample_data

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,2024-01-08,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,2024-01-09,375.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
